# Lab 06: Binary Operations in Python

Knowing how to work with binary in a Python environment will become very handy for cryptography applications in this course. This lab will walk you through many of the major operations required to encipher and decipher messages that are typically represented using binary systems.

## Part 1: Binary Operators

There are five binary operations that we're concerned with in this course. We'll cover them one at a time.

### AND (`&`)
The AND operation is a bitwise operation that compares corresponding bits in a binary number, and returns a 1 if both bits are 1. Otherwise AND returns 0. In Python, the symbol to perform a bitwise AND operation is `&`.

For example:

```
  01111 (15)
& 10001 (17)
-------
  00001 (1)
```

**Note:** Even though the `&` symbol is a bitwise operator, meaning that it works with the binary representation of a number to perform the intended operation, you can use it on any representation of a number in Python, such as decimal or hexadecimal. Python will just convert behind the scenes to perform the operations, and will always return the result in decimal format unless you instruct it otherwise.

Run the code cells below to verify this fact.

#### Decimal

In [1]:
15 & 17

1

#### Hexadecimal

In [2]:
0x0F & 0x11

1

### Question 1

#### Binary

In the cell below, compute the result of an AND operation between the decimal numbers 15 and 17, written as binary numbers.

In [3]:
0b01111 & 0b10001 # SOLUTION

1

### OR (`|`)

The OR bitwise operator uses the pipe symbol (`|`) in Python to compare corresponding bits in a number. It returns a 1 if either bit is 1, and returns a 0 if both bits are 0.

For example:

```
  01111 (15)
| 10001 (17)
-------
  11111 (31)
```

In [4]:
15 | 17

31

In [5]:
0x0F | 0x11

31

### Question 2

In the cell below, compute the result of an OR operation between the decimal numbers 15 and 17, written as binary numbers.

In [6]:
0b01111 | 0b10001 # SOLUTION

31

### XOR (`^`)

The XOR, or "exclusive OR", is a bitwise operator that uses the `^` character in Python. It compares corresponding bits of numbers, and returns a 1 if either bit is 1, and returns a 0 if both bits are the same (e.g. both 0 or both 1).

For example:

```
  01111 (15)
^ 10001 (17)
-------
  11110 (1)
```

In [7]:
15 ^ 17

30

In [8]:
0x0F ^ 0x11

30

### Question 3

In the cell below, compute the result of an XOR operation between the decimal numbers 15 and 17, written as binary numbers.

In [9]:
0b01111 ^ 0b10001 # SOLUTION

30

### Shift Operators

Bitwise shift operators are another kind of tool for bit manipulation. They let you move the bits around, which will be handy for certain operations later on. These types of operations are often used to improve the speed of certain mathematical operations since they are very low level operations on the computer and can be executed very quickly compared to using more modern programming languages.

#### Left Shift

The bitwise left shift operator (`<<`) moves the bits of its first operand (the number before the `<<`) to the left by the number of places specified in its second operand (the number after the `<<`). It also takes care of inserting enough zero bits to fill the gap that arises on the right edge of the new bit pattern:

![](lshift.gif)

For example, see how a single left shift to the number 100111 (39 in decimal) results in the number 1001110 (78).

```
a:         100111     39
a << 1:   1001110     78
```

and how two left shifts results in 10011100 (156).

```
a:         100111     39
a << 2:  10011100    156
```

Just like with the AND, OR, and XOR operations, Python can apply bitshifts to any representation of a number.

In [10]:
39 << 2

156

In [11]:
0x27 << 2

156

In [12]:
0b100111 << 2

156

### Question 4

In your own code cell, investigate and what happens to the value of numbers after left bitshifting them once, twice, and three times. Then, describe the mathematical operation that left bitshifts perform on numbers. Be as specific as you can in your description.

Each left bitshift doubles the value of the number. This makes sense because by shifting to the left, each bit is now representing a place value that is one power of two larger than it previously represented.

#### Right Shift

The bitwise right shift operator (`>>`) is analogous to the left one, but instead of moving bits to the left, it pushes them to the right by the specified number of places. The rightmost bits always get dropped:

![](rshift.gif)

For example:

```
a:      100111   39
a >> 1:  10011   19
```

and

```
a:      100111   39
a >> 2:   1001    9
```

In [13]:
39 >> 2

9

In [14]:
0x27 >> 2

9

In [15]:
0b100111 >> 2

9

## Question 5

In your own code cell, investigate and what happens to the value of numbers after **right** bitshifting them once, twice, and three times. Then, describe the mathematical operation that right bitshifts perform on numbers. Be as specific as you can in your description.

Halving an odd number such as $15711$ would produce a fraction. To get rid of it, the right shift operator automatically floors the result (rounds down to the nearest integer). It’s virtually the same as a floor division by a power of two:

$$ a >> n = \left\lfloor \frac{a}{2^n} \right\rfloor $$

That means in Python you can either shift a number 1 bit to the right, `a >> 1`, or floor divide by 2, `a // 2` to obtain the same result.

## Part 2: Bitmasks

A lot of our work with binary numbers will require us to inspect a binary number at the singular bit level. The next few questions will guide you through how to:
* **Get a bit**: Determine if there is a 1 or 0 at a particular location of a binary number
* **Set a bit**: Turn a particular bit of a number "on" to a 1
* **Clear a bit**: Turn a particular bit of a number "off" to a 0
* **Toggle a bit**: "Flip" a bit from one value to the other

The tool that we'll use to accomplish these operations are called **bitmasks**.

A **bitmask** works like a graffiti stencil that blocks the paint from being sprayed on particular areas of a surface. It lets you isolate certain bits, typically a single bit a time, so you can selectively work with only those selected bits. Bitmasking involves using both the bitwise logical operators and the bitwise shift operators covered previously in this lab. The name comes from an analogy to a face mask, which covers up some parts of your face and lets other parts show through. In computing, a bitmask covers up (or filters out) some bits in a binary number and allows others to pass through.

### Getting a Bit

Remember, the bitwise AND operation will compare corresponding bits in two numbers and return:
* a 0 if both bits are 0,
* a 0 if one bit is 0 and one bit 1, and
* a 1 if both bits are 1

In short, the only time AND will return a 1 is if both numbers have a 1 in the same position.

Suppose you wanted to create a function that takes in a number, and returns only the value of a specified bit.  We can use a bitmask to create a method to read the value of a particular bit in a number.

To read the value of a particular bit at a given position, you can use the bitwise AND (`&` operator) against a bitmask composed of all 0's except for one bit set to 1 at the desired location. The resulting number will have a 0 anywhere the bitmask had a 0, and retain the value of the bits anywhere the bitmask had a 1.

#### Example #1
What's the value of the 5th bit in `10110000`?


```
 number: 10110000
bitmask: 00010000  <-- will return value of bit at b_5
-----------------
result:  00010000
            ^ (the bit at b_5 was a 1)
```

We can see visually that the 5th bit is a 1, but how could the computer return that same value? Applying the bitmask `00010000` with the `&` operator returns `00010000` which is not 1, it's the decimal number 16. However, since know that all the bits to the right and the left of the 1 in the result will always be a 0 (because the bitmask also has 0's in those bits), if we left bitshift the result 4 spaces over, the result will be the decimal number 1.

```
     result: 00010000 (16)
result >> 4:     0001 (1)
```

This final result, `0001` or 1, is the exact value of the bit that was stored in bit #5 of the original number, `10110000`.

In [16]:
(0b10110000 & 0b00010000) >> 4

1

#### Example #2
What's the value of the 6th bit in `11000000`?

```
 number: 11000000
bitmask: 00100000  <-- will return value of bit at b_6
-----------------
result:  00000000  
           ^ (the bit at b_6 was a 0)
```

If you were interested in just knowing the bit value, 0 or 1, and not keeping the rest of the output from the bitmask process, then all you need to do is right-shift the result (`>>`) the appropriate number of spaces.

```
 number: 11000000
bitmask: 00100000
-----------------
result:  00000000
result >> 5 : 000
```

And `000` is equivalent to 0, which was the value in the 6th bit of the original number.

In [17]:
(0b11000000 & 0b100000) >> 5

0

### Question 6

Write a function `get_bit` that takes in an integer (decimal or binary, remember Python treats them as the same) and the index of a bit in that number. The function should use the bitmask technique to return either a 0 or a 1 that matches the value at the indicated index in the provided number. **Remember, the right-most bit is called bit #1!**

You should first use the specified `bit_index` to create a bitmask using bitshift operators, then apply the bitmask to obtain an initial result, and then use bitshift operators to compute the final 0 or 1.

In [18]:
def get_bit(value, bit_index):
    bitmask = 1 << (bit_index - 1) # SOLUTION
    result = value & bitmask  # SOLUTION
    return result >> (bit_index - 1)  # SOLUTION

In [19]:
get_bit(0b10100001, bit_index=1)

1

In [20]:
get_bit(0b10100001, bit_index=2)

0

In [21]:
get_bit(0b10100001, bit_index=6)

1

In [22]:
get_bit(0b10100001, bit_index=8)

1

### Setting a Bit

Defining a particular bit in a number to have a value of 1 is called *setting a bit*, and the operation similar to getting the value of a bit. You can use a bitmask, but instead of using bitwise AND, you use the bitwise OR (`|`) operator.

Think about why this would be true. The bitwise OR operator will compare corresponding bits in two numbers and return:
* a 0 if both bits are 0, 
* a 1 if one bit is 0 and the other bit is 1, 
* a 1 both bits are 1. 

So if you wanted to set the bit at the 4th bit ($b_4$) of the number `00010` to a 1, you could apply the bitmask `01000` and the OR operator:

```
number:  00010
bitmask: 01000 <-- b_4's value is 1, rest of bits are 0
--------------
result:  01010 (10 in decimal)
          ^ set to a 1 using the OR operation
```

In [23]:
0b00010 | 0b01000

10

You'll notice that because of how the OR operator works, 
* if the original number has a 0 at a particular bit, it will stay a 0 in the result if the bitmask has a 0 in the same position
* if the original number has a 1 at a particular bit, it will stay a 1 in the result if the bitmask has a 0 in the same position
* if the original number has a 1 at a particular bit, it will stay a 1 in the result if the bitmask has a 1 in the same position
* if the original number has a 0 at a particular bit, it will become a 1 in the result if the bitmask has a 1 in the same position

So the only bits that will change in the original number are 0's, when the bitmask has a 1 in the same position. That's why this combination of operations is perfect for "setting" a bit in a binary number.

### Question 7

Write a function, `set_bit` that will set take in an integer and a the index of a bit in that number. The function should apply a bitmask that sets the bit at the specified bit to 1, and return that result.

You should first use the specified `bit_index` to create a bitmask using bitshift operators, then apply the bitmask to obtain the result.

In [24]:
def set_bit(value, bit_index):
    bitmask = 1 << (bit_index - 1) # SOLUTION
    result = value | bitmask  # SOLUTION
    return result  # SOLUTION 

In [25]:
set_bit(0b10, bit_index = 4)

10

In [26]:
set_bit(0b10000000, bit_index=6)

160

In [27]:
set_bit(0b10000000, bit_index=5)

144

In [28]:
set_bit(0b10000000, bit_index=3)

132

In [29]:
set_bit(0b10000000, bit_index=1)

129

### Unsetting a Bit

Defining a bit to have a value of 0 is called *unsetting a bit* or *clearing a bit*. You can use a bitmask and the bitwise AND (`&`) operator to accomplish this task.

```
number:  10010
bitmask: 11101 <-- b_2's value is 0, rest of bits are 1
--------------
result:  10000  (16 in decimal)
            ^ unset to a 0 using the AND operation
```

Notice how this is very similar to setting a bit, only the bitmask is inverted. That is, all the 0's are now 1's, and vice versa.

In [30]:
0b10010 & 0b11101

16

Building an inverted bitmask can be a bit tricky, but fortunately Python has a built in "binary inverse" operator, the `~`. The invert (`~`) operation will invert an integer's binary form.

In [31]:
0b10010 & ~(0b00010)

16

### Question 8

Write a function, `clear_bit` that will set take in an integer and an index of a bit in that number. The function should apply a bitmask that unsets the bit at the indicated index to 0, and return that result.

In [32]:
def clear_bit(value, bit_index):
    bitmask = 1 << (bit_index - 1) # SOLUTION
    result = value & ~bitmask  # SOLUTION
    return result  # SOLUTION 

In [33]:
clear_bit(0b11111111, bit_index=6)

223

In [34]:
clear_bit(0b11111111, bit_index=4)

247

In [35]:
clear_bit(0b11111111, bit_index=3)

251

In [36]:
clear_bit(0b11111111, bit_index=1)

254

### Toggling a Bit

Lastly, sometimes you'll just want to toggle a bit from a 1 to a 0 or vice versa. We can accomplish that using a bitmask with a bitwise XOR (`^`) operation. Remember, the bitwise XOR operation returns:

* a 0 if the corresponding bits are the same (both 0's or both 1's)
* a 1 if the corresponding bits are different (one is a 0 and the other is a 1)

#### Example #1
```
number:  10010
bitmask: 00100 <-- b_3's value is 1, rest of bits are 0
--------------
result:  10110 (decimal value of 22)
           ^   toggled using a 1 in the bitmask with the XOR operation
```

In [37]:
0b10010 ^ 0b00100

22

#### Example #2
```
number:  10010
bitmask: 10000 <-- b_5's value is 1, rest of bits are 0
--------------
result:  00010 (decimal value of 2)
         ^     toggled using a 1 in the bitmask with the XOR operation
```

In [38]:
0b10010 ^ 0b10000

2

### Question 9

Write a function, `toggle_bit` that will set take in an integer and an index of a bit in that number. The function should apply a bitmask that toggles the bit at the indicated index and return the result.

In [39]:
def toggle_bit(value, bit_index):
    bitmask = 1 << (bit_index - 1) # SOLUTION
    result = value ^ bitmask  # SOLUTION
    return result  # SOLUTION 

In [40]:
toggle_bit(0b10010, bit_index=3)

22

In [41]:
toggle_bit(0b10010, bit_index=3)

22

In [42]:
toggle_bit(0b10010, bit_index=5)

2

# Submitting your work
You're done with this Lab! All assignments in the course will be distributed as notebooks like this one, and you will submit your work by doing the following:
* Save your notebook
* Restart the kernel and run up to this cell.
* Run all the tests by running the cell containing `grader.check_all()`. Make sure they pass the way you expect them to.
* Run the cell below with the code `grader.export(...)`.
* Download the file named `labXX.zip`, found in the explorer pane on the left side of the screen.
* Upload `labXX-<date-time stamp>.zip` to the corresponding lab assignment on Canvas.